# 04_m5_optional_validation
Phần tùy chọn trên Validation: Optuna LightGBM (T01), Croston SBA (X01), ablation LightGBM bỏ giá/sự kiện (X02), backtest OOF cho stacking (T02, trừ CatBoost). Không đọc nhãn Test. Kaggle: thêm dữ liệu M5 vào Input, bật Internet.

In [ ]:
import os, pathlib, subprocess, sys, shutil
BRANCH = "m5-end-to-end"
REPO = "https://github.com/haiphamt/IS403.R11-M5-Competition.git"
WORK = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle/working") else "/content")
os.chdir(WORK)  # không đứng trong thư mục repo sắp bị xóa
if (WORK / "repo").exists():
    shutil.rmtree(WORK / "repo")
subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(WORK / "repo")], check=True)
os.chdir(WORK / "repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsforecast", "catboost", "lightgbm", "xgboost", "scikit-learn", "pyarrow", "PyYAML", "optuna"], check=True)
OUT = WORK / "opt"
def run(*cmd):
    subprocess.run([sys.executable, *cmd], check=True)

In [ ]:
cpus = os.cpu_count()
run("scripts/tune_lightgbm.py", "--out", str(OUT / "tune"), "--n-jobs", str(cpus))
run("scripts/run_experiment.py", "--split", "validation", "--models", "croston_sba", "lightgbm_noprice", "lightgbm_tuned",
    "--lgbm-params", str(OUT / "tune" / "best_params.json"), "--workers", str(max(1, cpus - 1)), "--n-jobs", str(cpus), "--out", str(OUT / "runs"))
run("scripts/backtest_oof.py", "--models", "linear_regression", "lightgbm", "lightgbm_tuned", "xgboost",
    "--lgbm-params", str(OUT / "tune" / "best_params.json"), "--origins", "1829", "1857", "--n-jobs", str(cpus), "--out", str(OUT / "oof"))

In [ ]:
import glob
for d in glob.glob(str(OUT / "**" / "ckpt"), recursive=True):
    shutil.rmtree(d, ignore_errors=True)
print(shutil.make_archive(str(WORK / "m5_optional_validation"), "zip", str(OUT)))